# CIDS · 10 — S3d: learned jury (smarter choice of SegResNet's candidates)

S3b's rule `segjury_0.7` (keep a candidate if SAM agrees ≥ 70%) gives **test Dice 0.699**; keeping exactly the right candidates would give **~0.741**. S2c showed SAM-S2b already ranks real tumours above false blobs (AUC ~0.82) — the information is there, the hand-made rule just doesn't use it well.

**S3d** replaces the rule with a small classifier (logistic regression / gradient boosting) on per-candidate clues that exist at test time: size, rank, SegResNet confidence, SAM–SegResNet agreement, SAM hard + **soft** volume.

| step | what | time (T4) |
|---|---|---|
| train (1080) | pipeline → candidate clues + label (touches tumour?) → fit the 2 classifiers | ~1.5 h |
| val (120) | pick classifier + probability threshold by val Dice | ~10 min |
| test (306) | apply once; Dice / HD95 / NSD for learned vs segjury_0.7 vs seg_rel20, Wilcoxon, per collection | ~45 min |

Same models as S3b: **SegResNet-100 + SAM-S2b**. Fully automatic, no ground truth at test time.

**Settings:** GPU T4 x2 · Internet ON · Persistence: Files. **Inputs:** cache · `cids-checkpoints`.
Resumable: finished splits are cached (`<split>_features.csv`) — re-run the cell to continue.

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
CKPT_DIR   = '/kaggle/input/datasets/kumarvemuri1/cids-checkpoints'
SEG_CKPT   = f'{CKPT_DIR}/segresnet_img_e100_best.pt'
SAM_CKPT   = f'{CKPT_DIR}/sam_s2b_best.pt'
CODE_DIR   = '/kaggle/working/cids'
OUT        = '/kaggle/working/s3d_jury'
PATHS = [CACHE_ROOT, SEG_CKPT, SAM_CKPT]

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not).
    Progress bars redraw a line with '\r'; only each line's final state is printed (one line per epoch)."""
    import subprocess
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf = b''
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        buf += chunk
        *lines, buf = buf.split(b'\n')
        for line in lines:
            print(line.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if buf.strip():
        print(buf.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')


In [ ]:
sh(f"""pip install -q monai medim einops appdirs""")
sh(f"""pytest -q tests""")

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS)

## Smoke test (~5 min): 10 cases per split
Check: it prints `train: N candidates = x true + y false`, a `val logreg` / `val gboost` line, and a TEST table. Numbers on 10 cases mean nothing.

In [ ]:
sh(f"""python scripts/s3d_learned_jury.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_ckpt {SAM_CKPT} --out /kaggle/working/s3d_smoke --limit 10""")
sh(f"""rm -rf /kaggle/working/s3d_smoke""")

## Full run (~2.5 h)

In [ ]:
sh(f"""python scripts/s3d_learned_jury.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_ckpt {SAM_CKPT} --out {OUT}""")

In [ ]:
import json
print(json.dumps(json.load(open(f'{OUT}/summary.json')), indent=2))
sh(f"""ls -la {OUT}""")